In [ ]:
import pandas as pd
import requests
import os
import re

os.environ["HF_TOKEN"] = "SECRET_KEY"

# 1. Download dataset nguyên bản: tmquan/vbpl-vn

Vì dataset này đã qua xử lý rất nhiều, khó hợp nhất lại được nội dung gốc, nên ta sẽ sử dụng api_url tham khảo từ đây để kéo dữ liệu từ nguồn về dùng.

In [2]:
from tqdm.auto import tqdm 
from datasets import load_dataset

# 2. Lấy lại api_url từ dataset gốc
ds = load_dataset("tmquan/vbpl-vn", split="train", token=os.environ["HF_TOKEN"])


Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

In [3]:
import re
import pandas as pd
import numpy as np
import json


def _normalize_doc_number(doc_numbers):
    """
    [" 3131/QĐ-UB-NCVX ", "12 / TT-BTC"]
    -> "3131/QĐ-UB-NCVX, 12/TT-BTC"
    """

    if doc_numbers is None:
        return None
    if not isinstance(doc_numbers, list) :
        doc_numbers = [doc_numbers]

    cleaned = []

    for x in doc_numbers:
        if x is None:
            continue

        # xóa toàn bộ khoảng trắng
        x = re.sub(r"\s+", "", str(x))

        if x:
            cleaned.append(x)

    return ", ".join(cleaned)

def format_vbpl_dataset(dataset):
    """
    Parameters
    ----------
    dataset : datasets.Dataset

    Returns
    -------
    pandas.DataFrame
    """

    rows = []

    for sample in dataset:
        try:
            rows.append({
                "id": sample.get("item_id", np.nan),
                "scope": sample.get("scope"),
                "subject_title": sample.get("legal_area"),
                "issue_date": sample.get("issue_date"),
                "source_url": sample.get("source_url"),
                "api_url": sample.get("api_url"),
                "legal_type": sample.get('legal_type'),
                "legal_area": sample.get('legal_area'),
                "doc_number": _normalize_doc_number(
                    sample.get("doc_number")
                ),
                "title": sample.get("title"),
            })
        except:
            pass

    return pd.DataFrame(rows)

In [4]:
df = format_vbpl_dataset(ds)
df.head(5)

,id,scope,subject_title,issue_date,source_url,api_url,legal_type,legal_area,doc_number,title
0,1,trung_uong,Chưa phân loại,1950-03-27,https://vbpl.vn/van-ban/chi-tiet/nghi-dinh-so-...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Nghị định,Chưa phân loại,24/LĐ-NĐ,Tổ chức các cơ quan Lao động địa phương liên k...
1,10,trung_uong,Chưa phân loại,1950-10-16,https://vbpl.vn/van-ban/chi-tiet/thong-tu-so-4...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Thông tư,Chưa phân loại,41-NV-6-TT,Định thể lệ xếp công chức vào thang lương chun...
2,100,trung_uong,Chưa phân loại,1950-05-14,https://vbpl.vn/van-ban/chi-tiet/sac-lenh-so-6...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Sắc lệnh,Chưa phân loại,68/SL,thành lập Ban Kinh tế Chính phủ
3,1000,trung_uong,Chưa phân loại,1957-01-22,https://vbpl.vn/van-ban/chi-tiet/nghi-quyet-so...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Nghị quyết,Chưa phân loại,Khôngsố,Về việc hoàn toàn tín nhiệm Chính phủ
4,10000,trung_uong,Chưa phân loại,1995-01-25,https://vbpl.vn/van-ban/chi-tiet/chi-thi-so-64...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Chỉ thị,Chưa phân loại,64-TTg,"Về tăng cường công tác giải quyết khiếu nại, t..."


# 2. Re-Crawl Dataset 

In [20]:
from bs4 import BeautifulSoup

REDO_INDICES = []
REQUEST_ERROR = []
NON_ORGANIZED_DOCS = []

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/xml, text/xml, */*; q=0.01'
}


def parse_docs(docs_content: str) -> list:
    """
    Lược bỏ HTML dư thừa, chỉ giữ lại nội dung từ các thẻ <p> có class bắt đầu bằng 'prov-'.
    Chuyển đổi thành định dạng: <hậu_tố>Nội dung...
    """

    soup = BeautifulSoup(docs_content, "html.parser")
    parsed_lines = []
    
    # Tìm tất cả các thẻ <p> trong HTML
    for p in soup.find_all("p"):
        classes = p.get("class", [])
        
        # Tìm class đầu tiên bắt đầu bằng 'prov-' (đề phòng 1 thẻ có nhiều class)
        prov_class = next((c for c in classes if isinstance(c, str) and c.startswith("prov-")), None)
        
        if prov_class:
            # Lấy hậu tố sau chữ 'prov-' (VD: 'article' từ 'prov-article')
            suffix = prov_class.replace("prov-", "")
            
            # get_text() tự động bỏ các thẻ lồng bên trong (như b, strong, i, br...)
            # separator=" " giúp các thẻ sát nhau khi bị loại bỏ không bị dính chữ
            raw_text = p.get_text(separator=" ", strip=True)
            
            # Làm sạch khoảng trắng thừa
            clean_text = re.sub(r"\s+", " ", raw_text).strip()
            
            if clean_text:
                # Format đúng theo yêu cầu: <hậu_tố>Nội dung
                parsed_lines.append((suffix, clean_text))
                
    # Trả về chuỗi kết quả được nối bằng ký tự xuống dòng
    # print(parsed_lines)
    return parsed_lines


def parse_document(soup: BeautifulSoup) -> dict:
    
    # Kiểm tra xem thẻ <data> có tồn tại không
    data_tag = soup.find("data")
    if not data_tag:
        return None

    # 1. Trích xuất các trường từ gốc <data>
    # Dùng find(recursive=False) để chỉ lấy thẻ con trực tiếp, tránh trùng với id của documentContent/documentIssues...
    doc_id = (
        data_tag.find("id", recursive=False).text
        if data_tag.find("id", recursive=False)
        else None
    )
    docs_code = (
        data_tag.find("docNum").text if data_tag.find("docNum") else None
    )
    article_title = (
        data_tag.find("title").text if data_tag.find("title") else None
    )
    issue_date = (
        data_tag.find("issueDate").text if data_tag.find("issueDate") else None
    )
    eff_from = (
        data_tag.find("effFrom").text if data_tag.find("effFrom") else None
    )
    agency_name = (
        data_tag.find("agencyName").text if data_tag.find("agencyName") else None
    )

    # Lấy status từ effStatus -> name
    eff_status_tag = data_tag.find("effStatus")
    status = (
        eff_status_tag.find("name").text
        if eff_status_tag and eff_status_tag.find("name")
        else None
    )
    if status in ['Hết hiệu lực toàn bộ', 'Không còn phù hợp', 'Ngưng hiệu lực'] :
        return None

    # 2. Trường documentContent -> content (content_text)
    doc_content_tag = data_tag.find("documentContent")
    content_text = (
        doc_content_tag.find("content").text
        if doc_content_tag and doc_content_tag.find("content")
        else None
    )
    if content_text == None :
        return None
    
    content_text = parse_docs(content_text)
    if not content_text :
        NON_ORGANIZED_DOCS.append(doc_id)
        return None

    # 3. Trường documentMajors -> name (topic_title)
    doc_majors_tag = data_tag.find("documentMajors")
    topic_title = (
        doc_majors_tag.find("name").text
        if doc_majors_tag and doc_majors_tag.find("name")
        else None
    )

    # 4. Trường documentFields -> name (subject_title)
    doc_fields_tag = data_tag.find("documentFields")
    subject_title = (
        doc_fields_tag.find("name").text
        if doc_fields_tag and doc_fields_tag.find("name")
        else None
    )

    # 5. Trường references (Nằm trong thẻ <references>)
    # Trích xuất danh sách các tuple (id, docNum) từ targetDocument
    references = []
    references_tag = data_tag.find("references")
    if references_tag:
        # Tìm tất cả các thẻ <references> con bên trong
        for ref in references_tag.find_all("references"):
            target_doc = ref.find("targetDocument")
            if target_doc:
                ref_id = (
                    target_doc.find("id").text
                    if target_doc.find("id")
                    else None
                )
                ref_doc_num = (
                    target_doc.find("docNum").text
                    if target_doc.find("docNum")
                    else None
                )
                if ref_id or ref_doc_num:
                    references.append({
                        'id' : ref_id, 
                        'docs_code' : ref_doc_num
                    })

    # Tổng hợp thành dictionary
    result_dict = {
        "id": doc_id,
        "docs_code": _normalize_doc_number(docs_code),
        "docs_title": article_title,
        "issueDate": issue_date,
        "effFrom": eff_from,
        "status": status,
        "agency_name": agency_name,
        "topic_title": topic_title,
        "subject_title": subject_title,
        "references": references,  # Lưu dưới dạng list của các tuple
        "content_text": content_text,
    }

    return result_dict

def crawl_work(row:pd.Series):
    item_id = str(row["id"])
    api_url = str(row['api_url'])

    # print(item_id)
    
    if not api_url or api_url == 'nan':
        if not item_id :
            REDO_INDICES.append(row.name)
        else :
            api_url = f'https://vbpl-bientap-gateway.moj.gov.vn/api/qtdc/public/doc/{item_id}'

    try:
        # Timeout 10s để tránh treo vĩnh viễn
        res = requests.get(api_url, headers=HEADERS, timeout=30) 
        res.raise_for_status()
        
        xml_soup = BeautifulSoup(res.content, 'xml')

        results = parse_document(xml_soup)
        
        if results is not None :
            results['legal_type'] = str(row.get('legal_type', "Chưa xác định"))
            results['source_links'] = api_url

        return results
    except requests.exceptions.Timeout:
        REDO_INDICES.append(row.name)
        # print(f"Timout at {row.name}")
        return None
    except requests.exceptions.HTTPError as http_err:
        # Xử lý các lỗi HTTP không OK (ví dụ: 400, 401, 403, 404, 500...) trừ mã 304 đã bắt ở trên
        print(f"[ERROR] Lỗi HTTP xảy ra: loc={row.name} (Status code: {res.status_code})")
        REQUEST_ERROR.append(row)
        return None
    # except Exception:
    #     REDO_INDICES.append(row.name)
    #     print(f"[ERROR] at loc={row.name}")
    #     return None



In [6]:
i=15

print(df.loc[i]['doc_number'])
result = crawl_work(df.loc[i])

result['content_text'] if result is not None else ''

867/QĐ-QLTA


<unknown>:1: SyntaxWarning: invalid escape sequence '\s'


''

In [29]:
import pandas as pd

crawled_records = []

# Biến đếm để hiển thị trên thanh tqdm
success_count = 0
fail_count = 0
article_count = 0

MAX_WORKERS = 8


# Gộp danh sách thành chuỗi Regex
regex_pattern = "|".join(['Luật', 'Bộ luật', 'Hiến pháp', 'Pháp lệnh', 'Nghị quyết', 'Nghị quyết liên tịch', 'Nghị định', 'Thông tư', 'Thông tư liên tịch', 'Thông tư liên bộ', 'Quyết định', 'Văn bản hợp nhất', 'Công văn', 'Hiệp định', 'Nghị định thư', 'Chưa xác định'])

# # Init danh sách task
df_left = df[
    (df['scope'] != 'dia_phuong')                   # Lọc tài liệu địa phương   
    & (~df['id'].astype(str)
                .str.startswith("vbpqta")           # Lọc các bản dịch tiếng anh
                )
    & (df["legal_type"].fillna("Chưa xác định")     # Lọc một số dạng văn bản mang tính tạm thời (sắc lệnh, chỉ đạo,...)
                    .str.contains(regex_pattern)
                    )
    ]

In [30]:
df_left.loc[df_left['doc_number'] == '47/VBHN-BTC', 'api_url'] = 'https://vbpl-bientap-gateway.moj.gov.vn/api/qtdc/public/doc/146039'
df_left.loc[df_left['doc_number'] == '02/2025/NĐ-CP', 'api_url'] = 'https://vbpl-bientap-gateway.moj.gov.vn/api/qtdc/public/doc/184147'
df_left.loc[df_left['doc_number'] == '21/2015/NĐ-CP', 'api_url'] = 'https://vbpl-bientap-gateway.moj.gov.vn/api/qtdc/public/doc/52378'
df_left.loc[df_left['doc_number'] == '248/2025/QH15', 'api_url'] = 'https://vbpl-bientap-gateway.moj.gov.vn/api/qtdc/public/doc/26f6a970-685e-11f1-a80f-235b0dd78960'
df_left.loc[df_left['doc_number'] == '121/2025/TT-BTC', 'api_url'] = 'https://vbpl-bientap-gateway.moj.gov.vn/api/qtdc/public/doc/6776a0a0-543a-11f1-9456-0b6fbb8f8274'
df_left.loc[df_left['doc_number'] == '25/2012/QH13', 'api_url'] = 'https://vbpl-bientap-gateway.moj.gov.vn/api/qtdc/public/doc/70838'

REQUEST_ERROR = []

In [31]:
from concurrent import futures as thread_futures
from tqdm.notebook import tqdm

print(f"""
    Tiến hành crawl {len(df_left)} docs.
    Chú thích
        OK       : số docs đã tải về.
        Articles : số lượng Điều luật đã tải về tương ứng.
        Fail     : số lượng docs đã bỏ qua (do Timeout, Tài liệu rỗng, hoặc Hết hiệu lực).
        REDO     : số lượng docs có thể thử tải lại do Timeout.
        Error    : số lượng lỗi do HTTP request
    """)

for idx in range(3) :
    if len(df_left) == 0 : break

    REDO_INDICES = []

    with thread_futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # 2. Truyền row (kiểu pd.Series) vào hàm. Lúc này row.name sẽ là chỉ số iloc
        futures = [
            executor.submit(crawl_work, row) for _, row in df_left.iterrows()
        ]

        # 3. Tạo thanh tiến trình với cấu hình postfix ban đầu
        pbar = tqdm(
            thread_futures.as_completed(futures), total=len(futures), desc=f"Crawling {idx}"
        )

        for future in pbar:
            result = future.result()

            if result is not None:
                success_count += 1
                article_count += len(result['content_text'])
                crawled_records.append(result)
            else:
                fail_count += 1

            # 4. Cập nhật postfix theo thời gian thực
            pbar.set_postfix(OK=success_count, Articles=article_count, Fail=fail_count, Redo=len(REDO_INDICES), Error=len(REQUEST_ERROR))

    # 5. Chuyển list dict thành DataFrame hoàn chỉnh
    crawled_df = pd.DataFrame(crawled_records)
    df_left = df.loc[REDO_INDICES]

df_left = pd.concat([df_left, pd.DataFrame(REQUEST_ERROR)], ignore_index=False)


    Tiến hành crawl 41439 docs.
    Chú thích
        OK       : số docs đã tải về.
        Articles : số lượng Điều luật đã tải về tương ứng.
        Fail     : số lượng docs đã bỏ qua (do Timeout, Tài liệu rỗng, hoặc Hết hiệu lực).
        REDO     : số lượng docs có thể thử tải lại do Timeout.
        Error    : số lượng lỗi do HTTP request
    


Crawling 0:   0%|          | 0/41439 [00:00<?, ?it/s]

[ERROR] Lỗi HTTP xảy ra: loc=2914 (Status code: 400)
[ERROR] Lỗi HTTP xảy ra: loc=81026 (Status code: 400)
[ERROR] Lỗi HTTP xảy ra: loc=81348 (Status code: 400)
[ERROR] Lỗi HTTP xảy ra: loc=126825 (Status code: 400)


Crawling 1:   0%|          | 0/2 [00:00<?, ?it/s]

In [32]:
df_left

,id,scope,subject_title,issue_date,source_url,api_url,legal_type,legal_area,doc_number,title
2914,103193,trung_uong,Chưa phân loại,NaN,https://vbpl.vn/van-ban/chi-tiet/thong-tu-so-3...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Thông tư,Chưa phân loại,NaN,thong-tu-so-30-2014-tt-bca-quy-dinh-ve-bieu-ma...
81026,185704,trung_uong,Chưa phân loại,2025-12-29,https://vbpl.vn/van-ban/chi-tiet/thong-tu-so-5...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Thông tư,Chưa phân loại,59/2025/TT-BXD,"sửa đổi, bổ sung một số điều của các Thông tư ..."
81348,186024,trung_uong,Chưa phân loại,2026-01-05,https://vbpl.vn/van-ban/chi-tiet/van-ban-hop-n...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Văn bản hợp nhất,Chưa phân loại,02/VBHN-BTС,"Quy định việc quản lý đối với tiền mặt, giấy t..."
126825,7618,trung_uong,Chưa phân loại,NaN,https://vbpl.vn/van-ban/chi-tiet/quyet-dinh-so...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Quyết định,Chưa phân loại,NaN,quyet-dinh-so-125-1998-qd-ttg-ve-viec-thanh-la...


# 3. Checkpoint

In [ ]:
from src.preprocess_parquet import save

save(crawled_df, "data/crawl/DONE_CRAWL.parquet")

crawled_df.sample(5).to_json("data/crawl/SAMPLE_CRAWL.json", orient="records", force_ascii=False, indent=4)

if len(df_left) > 0 :
    df_left.to_parquet("data/crawl/FAIL_CRAWL.parquet")

In [2]:
from src.preprocess_parquet import load

crawled_df = load("data/crawl/DONE_CRAWL.parquet")

crawled_df.head(2)

,id,docs_code,docs_title,issueDate,effFrom,status,agency_name,topic_title,subject_title,references,content_text,legal_type,source_links
0,10020,51/TTLB,Thông tư liên tịch số 51/TTLB Quy định chế độ ...,1994-06-06T00:00:00,1994-06-06T00:00:00,Còn hiệu lực,Bộ Tài chính,NaN,Chưa phân loại,"[{'id': '31147', 'docs_code': '06/CP'}, {'id':...","[[part, I. ĐỐI TƯỢNG THU VÀ MỨC THU], [section...",Thông tư liên tịch,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...
1,100048,18/2016/NĐ-CP,"Nghị định số 18/2016/NĐ-CP Sửa đổi, bổ sung mộ...",2016-03-18T00:00:00,2016-03-23T00:00:00,Còn hiệu lực,Chính phủ,Ngân hàng,NaN,"[{'id': '30473', 'docs_code': '53/2013/NĐ-CP'}...","[[article, Điều 1. Sửa đổi, bổ sung một số điề...",Nghị định,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...


# 4. Thực hiện khớp format và merge với Pháp Điển

In [4]:
from src.preprocess_parquet import restore_article
import re
import pandas as pd
from typing import List, Tuple, Dict, Any

# Khởi tạo Regex (Biên dịch sẵn để tối ưu hiệu suất)
RE_ARTICLE = re.compile(r'^Điều\s+(\d+[a-zA-ZđĐ]*)', re.IGNORECASE)
RE_CLAUSE = re.compile(r'^(\d+)\.')
RE_ITEM = re.compile(r'^([a-zA-ZđĐ])\)')

def correct_tag(original_tag: str, text: str) -> str:
    """
    Kiểm tra và hạ cấp (fallback) tag nếu text không khớp định dạng chuẩn.
    """
    # Nếu tag ban đầu là 'article', nhưng nội dung không phải là Điều
    if original_tag == 'article':
        if RE_ARTICLE.search(text):
            return 'article'
        elif RE_CLAUSE.search(text):
            return 'clause'     # Fallback xuống Khoản
        elif RE_ITEM.search(text):
            return 'item'       # Fallback xuống Điểm
        else:
            return 'content'    # Fallback cuối cùng nếu nó chỉ là text nối tiếp
            
    # Hỗ trợ thêm: Nếu tag là 'clause' nhưng lại chứa định dạng 'a)'
    if original_tag == 'clause':
        if RE_CLAUSE.search(text):
            return 'clause'
        elif RE_ITEM.search(text):
            return 'item'
        else:
            return 'content'
            
    return original_tag

def parse_legal_tuples(tuples_list: List[Tuple[str, str]]) -> List[Dict[str, Any]]:
    result = []
    
    # ---------------------------------------------------------
    # 1. QUẢN LÝ TRẠNG THÁI (STATE MANAGEMENT)
    # ---------------------------------------------------------
    state = {
        'current_part': None,         # Lưu số hiệu La Mã của Part gần nhất
        'active_root': None,          # Chứa Node gốc hiện tại (Article hoặc Section cấp 1)
        'active_sections': {},        # Dictionary lưu Section theo độ sâu {depth: node}
        'active_clause': None,        # Clause hiện tại đang mở
        'active_item': None           # Item hiện tại đang mở
    }

    # ---------------------------------------------------------
    # 2. CÁC HÀM XỬ LÝ TEXT (EXTRACTORS)
    # ---------------------------------------------------------
    def extract_part(text: str) -> str:
        """Trích xuất số La Mã từ part (Hỗ trợ 'I.', 'I -', 'Chương I.',...)"""
        match = re.search(r'^(?:Chương\s+)?([IVXLCDM]+)', text, re.IGNORECASE)
        return match.group(1).upper() if match else None

    def extract_article(text: str) -> str:
        """Trích xuất định dạng 'Điều 2', 'Điều 2a'"""
        match = re.search(r'^(Điều\s+\d+[a-zA-ZđĐ]*)', text, re.IGNORECASE)
        return match.group(1).capitalize() if match else "Điều"

    def extract_section(text: str) -> str:
        """Trích xuất số thứ tự của section (Hỗ trợ 1, 1.1, 1.1.1)"""
        match = re.search(r'^(\d+(?:\.\d+)*)', text)
        return match.group(1) if match else None

    # ---------------------------------------------------------
    # 3. CÁC HÀM XỬ LÝ CÂY (TREE HELPERS)
    # ---------------------------------------------------------
    def reset_children(level: str, section_depth: int = 0):
        """Xóa các mốc neo của các cấp độ thấp hơn khi có một mốc mới được tạo"""
        if level == 'root':
            state['active_sections'].clear()
            state['active_clause'] = None
            state['active_item'] = None
        elif level == 'section':
            # Xóa các section sâu hơn depth hiện tại (vd đang ở 1.2 thì xóa 1.2.1)
            keys_to_remove = [k for k in state['active_sections'].keys() if k > section_depth]
            for k in keys_to_remove:
                del state['active_sections'][k]
            state['active_clause'] = None
            state['active_item'] = None
        elif level == 'clause':
            state['active_item'] = None

    def get_deepest_parent(for_tag: str) -> Dict[str, Any]:
        """Tìm Node cấp sâu nhất đang mở để gắn nhánh con hoặc text vào"""
        if for_tag == 'content':
            if state['active_item']: return state['active_item']
            if state['active_clause']: return state['active_clause']
            if state['active_sections']: return state['active_sections'][max(state['active_sections'].keys())]
            if state['active_root']: return state['active_root']
            
        elif for_tag == 'item':
            if state['active_clause']: return state['active_clause']
            if state['active_sections']: return state['active_sections'][max(state['active_sections'].keys())]
            if state['active_root']: return state['active_root']
            
        elif for_tag == 'clause':
            if state['active_sections']: return state['active_sections'][max(state['active_sections'].keys())]
            if state['active_root']: return state['active_root']
            
        return None # Trả về None nếu hoàn toàn chưa có thẻ gốc nào được mở

    # ---------------------------------------------------------
    # 4. LUỒNG XỬ LÝ CHÍNH (MAIN PARSING LOOP)
    # ---------------------------------------------------------
    for tag, text in tuples_list:
        
        tag = tag.lower().strip()
        text = text.strip()
        
        # --- BỔ SUNG LỚP ĐÁNH CHẶN VÀ FALLBACK Ở ĐÂY ---
        tag = correct_tag(tag, text)
        
        # Bắt Part để lưu vết (Không push vào cây)
        if tag == 'part':
            part_idx = extract_part(text)

        # Bỏ qua các tag lạ
        if tag not in ['article', 'section', 'clause', 'item', 'content']:
            continue

        # --- XỬ LÝ ARTICLE ---
        if tag == 'article':
            idx_str = extract_article(text)
            node = {'title': text, 'content': []}
            result.append({
                'article_index': idx_str,
                'article_title': text,
                'content_text': node
            })
            state['active_root'] = node
            reset_children('root')

        # --- XỬ LÝ SECTION (VÀ NESTED SECTION) ---
        elif tag == 'section':
            idx_str = extract_section(text)
            node = {'title': text, 'content': []}
            
            # Xử lý fallback nếu section không bắt được số
            if not idx_str:
                prefix = f"Chương {state['current_part']}, " if state['current_part'] else ""
                result.append({
                    'article_index': f"{prefix}Mục",
                    'article_title': text,
                    'content_text': node
                })
                state['active_root'] = node
                reset_children('root')
                continue

            # Tính độ sâu dựa vào số lượng dấu chấm
            depth = len(idx_str.split('.'))

            if depth == 1: # Gốc cấp 1
                prefix = f"Chương {state['current_part']}, " if state['current_part'] else ""
                result.append({
                    'article_index': f"{prefix}Mục {idx_str}",
                    'article_title': text,
                    'content_text': node
                })
                state['active_root'] = node
                state['active_sections'][1] = node
                reset_children('section', 1)
                
            else: # Nested Section (Cấp > 1)
                # Tìm cha gần nhất của nó (vd: sâu 3 thì tìm 2 trước, không có thì tìm 1)
                parent = None
                for d in range(depth - 1, 0, -1):
                    if d in state['active_sections']:
                        parent = state['active_sections'][d]
                        break
                
                # Nếu không có section cha, bám vào root hiện tại
                if not parent and state['active_root']:
                    parent = state['active_root']
                    
                if parent:
                    parent['content'].append(node)
                else:
                    # Fallback nhảy cóc bất đắc dĩ (đẩy ra làm mốc độc lập)
                    prefix = f"Chương {state['current_part']}, " if state['current_part'] else ""
                    result.append({
                        'article_index': f"{prefix}Mục {idx_str}",
                        'article_title': text,
                        'content_text': node
                    })
                    state['active_root'] = node

                state['active_sections'][depth] = node
                reset_children('section', depth)

        # --- XỬ LÝ CÁC NHÁNH CON VÀ NỘI DUNG ---
        elif tag == 'clause':
            node = {'title': text, 'content': []}
            parent = get_deepest_parent('clause')
            if parent:
                parent['content'].append(node)
            state['active_clause'] = node
            reset_children('clause')

        elif tag == 'item':
            node = {'title': text, 'content': []}
            parent = get_deepest_parent('item')
            if parent:
                parent['content'].append(node)
            state['active_item'] = node

        elif tag == 'content':
            deepest_node = get_deepest_parent('content')
            if deepest_node:
                # Chỉ dồn nội dung nếu có một thẻ đang mở
                deepest_node['title'] = f"{deepest_node['title']} {text}".strip()
            # Yêu cầu #2: Nếu deepest_node = None, chuỗi text rơi vào khoảng không (bị xóa bỏ chuẩn chỉnh).

    # ---------------------------------------------------------
    # 5. POST-PROCESSING (CHUẨN HÓA ĐỊNH DẠNG CÂY)
    # ---------------------------------------------------------
    def format_tree(node: Dict[str, Any]) -> Dict[str, Any]:
        if not node['content']:
            return {'text': node['title']}
        else:
            node['content'] = [format_tree(child) for child in node['content']]
            return node

    for item in result:
        item['content_text'] = format_tree(item['content_text'])
        
    return result

def transform_legal_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    # 1. Đổi tên cột 'id' thành 'docs_vbpl_id'
    df = df.rename(columns={'id': 'docs_vbpl_id'})
    
    # 2. Xóa cột 'issuedDate' và 'effFrom' (Bao trùm cả lỗi gõ thiếu chữ 'd' nếu có)
    cols_to_drop = [col for col in ['issueDate', 'effFrom'] if col in df.columns]
    df = df.drop(columns=cols_to_drop)
    
    # 3. Apply hàm parse_legal_tuples vào cột 'content_text'
    # Lưu ý: Cấu trúc x phải là list. Nếu x đang là string (do load từ file), bạn cần dùng ast.literal_eval(x) hoặc json.loads(x)
    df['parsed_list'] = df['content_text'].apply(
        lambda x: parse_legal_tuples(x) if isinstance(x, list) else []
    )
    
    # Xóa cột content_text cũ đi vì chúng ta đã có parsed_list
    df = df.drop(columns=['content_text'])
    
    # 4. Explode DataFrame: Biến mỗi phần tử dict trong list thành một dòng riêng biệt
    df_exploded = df.explode('parsed_list', ignore_index=True)
    
    # Loại bỏ các dòng rỗng (trường hợp văn bản không có Điều/Mục nào bóc tách được)
    df_exploded = df_exploded.dropna(subset=['parsed_list'])
    
    # 5. Giải nén (unpack) các dictionary trong cột 'parsed_list' thành các cột riêng biệt
    df_exploded['article_index'] = df_exploded['parsed_list'].apply(lambda d: d.get('article_index'))
    df_exploded['article_title'] = df_exploded['parsed_list'].apply(lambda d: d.get('article_title'))
    df_exploded['content_text'] = df_exploded['parsed_list'].apply(lambda d: d.get('content_text'))
    
    # Xóa cột tạm
    df_exploded = df_exploded.drop(columns=['parsed_list'])
    
    # 6. Sắp xếp lại thứ tự cột cho dễ nhìn (Tùy chọn)
    ordered_cols = [
        'docs_vbpl_id', 'docs_code', 'docs_title', 'status', 'article_index', 'article_title', 
        'agency_name', 'topic_title', 'subject_title', 'legal_type',
        'content_text', 'references', 'source_links'
    ]

    # Lọc ra các cột thực sự tồn tại trong df để tránh lỗi
    final_cols = [col for col in ordered_cols if col in df_exploded.columns]
    df_final = df_exploded[final_cols]

    df_final['content_word_count'] = df_final['content_text'].apply(lambda x: len(restore_article(x))) 
    df_final['content_clause_count'] = df_final['content_text'].apply(lambda x: len(x.get('content', [])))
    
    return df_final


final_df = transform_legal_dataframe(crawled_df)

save(final_df, 'data/processed/vbpl_processed.parquet')
print(f"Tổng số mẫu vbpl: {len(final_df)}")

final_df.head()

Đã lưu dữ liệu vào data/processed/vbpl_processed.parquet
Tổng số mẫu vbpl: 211796


,docs_vbpl_id,docs_code,docs_title,status,article_index,article_title,agency_name,topic_title,subject_title,legal_type,content_text,references,source_links,content_word_count,content_clause_count
0,10020,51/TTLB,Thông tư liên tịch số 51/TTLB Quy định chế độ ...,Còn hiệu lực,Mục 1,1. Đối tượng thu:,Bộ Tài chính,NaN,Chưa phân loại,Thông tư liên tịch,"{'text': '1. Đối tượng thu: Các tổ chức, cá nh...","[{'id': '31147', 'docs_code': '06/CP'}, {'id':...",https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,216,0
1,10020,51/TTLB,Thông tư liên tịch số 51/TTLB Quy định chế độ ...,Còn hiệu lực,Mục 2,2. Mức thu:,Bộ Tài chính,NaN,Chưa phân loại,Thông tư liên tịch,{'text': '2. Mức thu: STT Loại hình thẩm định ...,"[{'id': '31147', 'docs_code': '06/CP'}, {'id':...",https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,1747,0
2,10020,51/TTLB,Thông tư liên tịch số 51/TTLB Quy định chế độ ...,Còn hiệu lực,Mục 1,1. Tổ chức thu:,Bộ Tài chính,NaN,Chưa phân loại,Thông tư liên tịch,"{'title': '1. Tổ chức thu:', 'content': [{'tex...","[{'id': '31147', 'docs_code': '06/CP'}, {'id':...",https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,614,2
3,10020,51/TTLB,Thông tư liên tịch số 51/TTLB Quy định chế độ ...,Còn hiệu lực,Mục 2,2. Quản lý và sử dụng nguồn thu lệ phí:,Bộ Tài chính,NaN,Chưa phân loại,Thông tư liên tịch,{'text': '2. Quản lý và sử dụng nguồn thu lệ p...,"[{'id': '31147', 'docs_code': '06/CP'}, {'id':...",https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,2326,0
4,100048,18/2016/NĐ-CP,"Nghị định số 18/2016/NĐ-CP Sửa đổi, bổ sung mộ...",Còn hiệu lực,Điều 1,"Điều 1. Sửa đổi, bổ sung một số điều của Nghị ...",Chính phủ,Ngân hàng,NaN,Nghị định,"{'title': 'Điều 1. Sửa đổi, bổ sung một số điề...","[{'id': '30473', 'docs_code': '53/2013/NĐ-CP'}...",https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,2741,3


In [10]:
def merge_new_dataset(df_target: pd.DataFrame, df_new: pd.DataFrame) -> pd.DataFrame:
    
    # 2. Lấy danh sách cấu trúc cột chuẩn từ DataFrame gốc
    target_columns = df_target.columns
    
    # 3. Ép cấu trúc df_new theo đúng target_columns
    # Hàm reindex sẽ:
    # - Giữ lại các cột khớp tên.
    # - Bỏ đi các cột dư thừa (như source_note_text, content_word_count, __index_level_0__,...).
    # - Tự động điền NaN cho các cột mà df_new đang thiếu (như docs_vbpl_id, status, agency_name,...).
    df_new = df_new.reindex(columns=target_columns)
    
    # 4. Nối 2 DataFrame lại với nhau
    # ignore_index=True giúp reset và đánh số index nối tiếp liền mạch từ 0 đến cuối
    df_combined = pd.concat([df_target, df_new], ignore_index=True)
    
    return df_combined


phapdien_unique_df = load('data/processed/phapdien_processed_unique.parquet')

combined_df = merge_new_dataset(
    df_target = final_df,
    df_new = phapdien_unique_df    
)

save(combined_df, 'data/processed/final_combine.parquet')
print(f"Tổng số mẫu combine: {len(combined_df)}")


Đã lưu dữ liệu vào data/processed/final_combine.parquet
Tổng số mẫu combine: 213272


# 5. Prunning các tài liệu có độ dài quá lớn (phụ lục)

In [ ]:
combined_df['mean_word_per_clause'] = (combined_df['content_word_count'] / (combined_df['content_clause_count'] + 1)).astype(int)

prunned_df = df.sort_values('mean_word_per_clause', ascending=False).iloc[300:]

save(prunned_df, "data/processed/combine_prunned.parquet")

prunned_df